# Phase 5 — Training

**Why (paper §3.7, §3.12):** we train the three quantile heads with the **pinball loss**
so each head learns its percentile, on the **log(AQI)** target (the label is right-skewed,
so a few extreme days would otherwise dominate). Settings mirror the paper: AdamW
(lr 3e-4), cosine schedule with 2 warm-up epochs, batch 32, ≤40 epochs with early
stopping, mixed precision.

> ⏳ This is the long step — on a Colab T4 GPU, expect ~20–40 minutes. Make sure the GPU
> is on (Runtime → Change runtime type → T4) and the physics cache from Phase 3 exists.

**Accuracy upgrades on by default (Phase 5b):** a dedicated Huber **point-head** for the
accuracy numbers, **class-balanced sampling** (more rare high-AQI photos), and more epochs —
all set in `configs/default.yaml`. These target the extreme-AQI underprediction from the first run.

## Bootstrap — run this first

This one cell makes the notebook self-contained: it grabs the code from GitHub (if it
isn't already here), installs the libraries, connects Google Drive, and makes our `src`
modules importable. **Set `REPO_URL` to your repository's URL.** It's safe to re-run and
also works on a laptop.

In [ ]:
# === Bootstrap — RUN ME FIRST (set REPO_URL to your repo) ===
REPO_URL = "https://github.com/YOUR_USERNAME/pm25-visual-aq.git"   # <-- EDIT THIS

import os, sys, subprocess

def _find_repo_root():
    # Are we already inside the repo (or just above the notebooks/ folder)?
    for cand in (".", "..", "pm25-visual-aq"):
        if os.path.isdir(os.path.join(cand, "src")):
            return os.path.abspath(cand)
    return None

_root = _find_repo_root()
if _root is None:                       # fresh Colab session: clone the code
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "pm25-visual-aq"], check=True)
    _root = os.path.abspath("pm25-visual-aq")
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=False)
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")

print("repo root:", _root, "| Colab:", IN_COLAB)

In [ ]:
import os, torch, matplotlib.pyplot as plt
from src.config import load_config
from src import data, splits, physics, dataset as D, model as M, train as T
cfg = load_config()
T.set_seed(cfg["seed"])
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

SOURCE = cfg["data"]["drive_path"]        # laptop test: "tests/fixture_ds"
STRATEGY = cfg["split"]["strategy"]        # station_grouped (primary)

ds, df = data.load_clean(SOURCE, from_disk=True, seed=cfg["seed"])
sp = splits.make_splits(df, strategy=STRATEGY, seed=cfg["seed"],
                        station_col=cfg["data"]["station_col"], time_col=cfg["data"]["time_col"],
                        lon_col=cfg["data"]["lon_col"], lat_col=cfg["data"]["lat_col"])
print("split:", STRATEGY, "->", sp["split"].value_counts().to_dict())

## Load the cached maps and build the data loaders

If the physics cache is missing, run `03_physics_features.ipynb` first.

In [ ]:
cache_path = os.path.join(cfg["data"]["cache_dir"], "physics_maps_%d.npy" % cfg["data"]["image_size"])
assert os.path.exists(cache_path), "Physics cache missing — run 03_physics_features.ipynb first."
cache = physics.load_map_cache(cache_path)

loaders = D.make_dataloaders(ds, sp, cache, cfg, num_workers=2)
print("batches:", {k: len(v) for k, v in loaders.items()})

## Train
The best model (lowest calibration loss) is saved to your Drive.

In [ ]:
net = M.build_model(cfg).to(device)
out_dir = os.path.join(cfg["data"]["outputs_dir"], STRATEGY)
history = T.train_model(net, loaders, cfg, device=device, out_dir=out_dir)
print("best epoch:", history["best_epoch"] + 1, "| best cal loss:", round(history["best_cal_loss"], 4))
print("checkpoint:", history["checkpoint"])

## Learning curves
Train and calibration loss should fall and then flatten; early stopping keeps the best.

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(history["train_loss"], label="train")
plt.plot(history["cal_loss"], label="calibration")
plt.axvline(history["best_epoch"], ls="--", c="grey", label="best")
plt.xlabel("epoch"); plt.ylabel("pinball loss (log-AQI)"); plt.legend()
plt.title("Training curves"); plt.show()

## What's next

We now have a trained model whose three outputs are *ordered* but **not yet guaranteed**
to contain the truth 90% of the time. **Next:** `06_calibrate_evaluate.ipynb` — conformal
calibration to earn that guarantee, then the full metrics.